# mEPSCAnalysis

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/mEPSCAnalysis.md`


Notebook source link: [mEPSCAnalysis.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/mEPSCAnalysis.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "mEPSCAnalysis"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"mEPSCAnalysis: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"mEPSCAnalysis: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"mEPSCAnalysis: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"mEPSCAnalysis: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# mEPSCAnalysis: synthetic current trace and event detection summary.
dt = 0.0005
time = np.arange(0.0, 12.0, dt)
n = time.size

# Generate baseline noise and negative-going mEPSC-like events.
trace = 0.025 * rng.standard_normal(n)
event_times_true = np.sort(rng.uniform(0.4, 11.6, size=75))
event_amps = rng.uniform(0.12, 0.42, size=event_times_true.size)
tau_rise = 0.0015
tau_decay = 0.010

kernel_t = np.arange(0.0, 0.060, dt)
kernel = (1.0 - np.exp(-kernel_t / tau_rise)) * np.exp(-kernel_t / tau_decay)
kernel = kernel / np.max(kernel)

for t_evt, amp in zip(event_times_true, event_amps, strict=False):
    idx = int(round(t_evt / dt))
    end = min(idx + kernel.size, n)
    k = kernel[: end - idx]
    trace[idx:end] -= amp * k

# Simple threshold-crossing detection with refractory period.
threshold = -0.12
refractory = int(round(0.006 / dt))
candidate = np.where(trace < threshold)[0]
detected_idx: list[int] = []
last = -refractory
for idx in candidate:
    if idx - last >= refractory:
        window_end = min(idx + int(round(0.004 / dt)) + 1, n)
        local = idx + int(np.argmin(trace[idx:window_end]))
        detected_idx.append(local)
        last = local
detected_idx = np.array(detected_idx, dtype=int)
detected_times = detected_idx * dt
detected_amps = -trace[detected_idx]
events = Events(times=detected_times, labels=[f"e{i}" for i in range(detected_times.size)])

fig, axes = plt.subplots(3, 1, figsize=(10, 7.2), sharex=False)
axes[0].plot(time, trace, color="0.15", linewidth=0.7)
axes[0].scatter(detected_times, trace[detected_idx], color="tab:red", s=10, alpha=0.8)
axes[0].set_title(f"{TOPIC}: synthetic mEPSC trace with detections")
axes[0].set_ylabel("current [a.u.]")

axes[1].hist(detected_amps, bins=25, color="tab:blue", alpha=0.85)
axes[1].set_title("Detected event amplitudes")
axes[1].set_xlabel("amplitude [a.u.]")
axes[1].set_ylabel("count")

iei = np.diff(events.times) if events.times.size > 1 else np.array([0.0])
axes[2].hist(iei, bins=25, color="tab:green", alpha=0.85)
axes[2].set_title("Inter-event interval distribution")
axes[2].set_xlabel("interval [s]")
axes[2].set_ylabel("count")

plt.tight_layout()
plt.show()

assert events.times.size > 30
assert float(np.mean(detected_amps) if detected_amps.size else 0.0) > 0.08
CHECKPOINT_METRICS = {
    "detected_event_count": float(events.times.size),
    "mean_detected_amplitude": float(np.mean(detected_amps) if detected_amps.size else 0.0),
}
CHECKPOINT_LIMITS = {
    "detected_event_count": (30.0, 220.0),
    "mean_detected_amplitude": (0.08, 0.6),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
